In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    # Open the real AI Assistant page (pinned sidebar button, verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'AI Assistant')]"))).click()
    chat_input = wait.until(EC.visibility_of_element_located((By.ID, "ai-chat-input")))
    send_btn = driver.find_element(By.XPATH, "//button[@type='submit' and contains(., 'Send')]")

    # Leave the query completely empty. The real UI disables Send on empty drafts
    # (verified: disabled={sending || !draft.trim()} in ChatInput.jsx).
    chat_input.clear()
    time.sleep(1)
    if not send_btn.is_enabled():
        print("Send button is disabled for an empty query (submission prevented).")
    else:
        send_btn.click()
        time.sleep(3)
        print("Send was enabled; clicked with an empty query.")

    # An empty query must not produce a normal AI response bubble
    bubbles = [el for el in driver.find_elements(By.XPATH, "//*[text()='AI Assistant']") if el.is_displayed()]
    assert len(bubbles) <= 1, "An AI response bubble appeared for an empty query."
    print("PASS: Empty AI query was handled correctly")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("40_ai_empty_query_FAIL.png")
finally:
    driver.quit()